# Chess — AlphaZero Training (control for ThompsonZero)

This notebook trains a **standard AlphaZero** agent on chess, as a **controlled
comparison** against `chess_thompson_zero_training.ipynb`. Its purpose is
diagnostic: if AlphaZero learns chess on this hardware and ThompsonZero
doesn't, the limitation is the Thompson search/target design; if *neither*
learns in comparable compute, the limitation is the hardware/scale (chess is
enormously harder than 6×6 Boop, and home-GPU AlphaZero chess is a large
undertaking even for the standard algorithm).

To make the comparison clean, **everything except the search algorithm and its
training targets is shared, verbatim, with the Thompson notebook**:

- the chess game (OpenSpiel native C++), observation planes, and action space;
- the network **body** — conv stem + SE-ResBlocks — at the *same* `CHANNELS`
  and `NUM_BLOCKS`, so the per-forward **body FLOPs are identical** (the fair
  hardware control). Only the heads differ (policy logits + scalar value here;
  a per-action Dirichlet + confidence in Thompson);
- every DirectML-safe primitive (elementwise GroupNorm/softplus, `LerpFreeAdamW`,
  the on-device gather probe, CPU replicas for batch-1 eval);
- the multiprocess self-play pool (N CPU workers + one GPU inference server,
  leapfrog half-batching, sparse legal-action wire format);
- the unified Elo eval pool (checkpoint × sim-budget), checkpoint/resume (with benchmark dedup),
  LR schedule, opponent pool + random-mover diversity, and the draw / buffer
  diagnostics.

### The algorithm (standard AlphaZero + KataGo optimizations)

Search is **PUCT MCTS**: each node keeps per-edge visit counts `N` and value
sums `W` (`Q = W/N`); selection maximizes `Q + c·P·√ΣN/(1+N)`; a leaf's scalar
network value is backed up the path, flipping sign every ply. Nothing
Bayesian — no belief distributions. Targets are the classic AlphaZero pair:

- **policy target** = the root's visit distribution over legal moves;
- **value target** = the game outcome `z` (mover's perspective).

On top of that baseline, this notebook includes the **KataGo optimizations
proven on Boop in the `kata_update` branch** — so it's the strongest standard
recipe, not a strawman:

- **MCTS-Solver** — exact WIN/DRAW/LOSS proof propagation (also in Thompson, so
  it isn't a confound), with proven positions emitted as exact
  policy+value training samples;
- **Forced playouts + policy-target pruning** (Wu 2020) — every noised root
  child gets at least `√(k·P·N)` visits so Dirichlet-seeded moves are actually
  explored, and the policy target then subtracts the forced visits the move
  didn't earn on merit (`K_FORCED`);
- **Value-target bootstrap** — `value = λ·z + (1−λ)·root_MCTS_value`
  (`VALUE_LAMBDA=0.5`, KataGo's default: the final `z` alone is near coin-flip
  noise on a balanced game). `VALUE_LAMBDA=1.0` recovers pure-`z` AlphaZero;
- **Playout-cap randomization** — a fraction of games use the full sim budget,
  the rest a fast budget (`FAST_MCTS_SIMS`/`FULL_MCTS_SIMS`);
- **Root Dirichlet noise** (`DIRICHLET_ALPHA`/`DIRICHLET_FRAC`) and
  **SE-ResBlocks** (in the shared body).

### Loss

Standard AlphaZero: **policy cross-entropy** (over legal moves — illegal logits
masked before the softmax so the target renormalizes over legal actions) +
**value mean-squared error**, `total = CE + VALUE_LOSS_WEIGHT·MSE`. Every op has
a DirectML kernel (`log_softmax`, `mse`, `mul`, `sum`), so — unlike the Thompson
notebook's lgamma/digamma losses — **no CPU split-backward is needed**.

### What to read off the run

The Ep line prints `pol=` (policy CE) and `val=` (value MSE) alongside the same
`dr=`/`cut=`/`ply=`/`buf=`/`aux=` self-play diagnostics as the Thompson
notebook. Evaluation uses the **same unified Elo pool as ThompsonZero** so the
two are directly comparable: every checkpoint enters ONE shared table as three
players — `@0` (search-free policy argmax), `@32`, `@128` (AZ-MCTS) — plus one
`random` mover. A new checkpoint's players each play `EVAL_GAMES_PER_PAIR` games
(alternating colours) against every existing player, then `EVAL_RANDOM_MULT·N`
random-pair games keep old ratings mixing. The decisive readouts: does the
current model climb above `random` and older checkpoints, and — the direct
analogue of the Thompson diagnostic — is the **search benefit** `Elo(@128) −
Elo(@0)` positive and growing? If the AlphaZero control learns where
ThompsonZero plateaus, the Thompson design is the bottleneck; if both plateau,
it's the problem's scale on this hardware.

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
import pyspiel

# Chess is already implemented natively in OpenSpiel (C++, games/chess/) — no
# custom engine needed here, unlike the Boop notebooks. It provides exact
# detection of every real chess draw (insufficient material, threefold
# repetition, stalemate, the 50-move rule) as well as checkmate, all inside
# is_terminal()/returns(); utilities are exactly WinUtility=+1, DrawUtility=0,
# LossUtility=-1, matching this notebook's v = p_win - p_loss value scale
# with no rescaling needed.
#
# Board-plane convention (see chess.cc ObservationTensor): the 20 planes are
# ABSOLUTE, not player-relative — White/Black pieces occupy fixed planes
# regardless of who is asking, with a side-to-play plane telling the network
# whose turn it is (unlike Boop's mover-relative flat observation). This is
# also why there is no board-symmetry augmentation available for chess (a
# left-right mirror is not a symmetry of the starting position: King/Queen
# occupy distinct files) — see the intro cell for a color-flip 180° rotation
# that WOULD be a valid symmetry, left unimplemented for now.
GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = GAME.num_distinct_actions()
_OBS_SHAPE   = tuple(GAME.observation_tensor_shape())    # (20, 8, 8)

print('Game:', GAME.get_type().long_name)
print('Actions:', _NUM_ACTIONS)
print('Observation shape:', _OBS_SHAPE)
print('Utility (min/max):', GAME.min_utility(), GAME.max_utility())

_state = GAME.new_initial_state()
print()
print(_state)
print('Legal actions from the start position:', len(_state.legal_actions()))
del _state

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# Same policy as the Boop notebooks: the GPU only wins with BIG batches, so
# self-play NN requests are funneled through one batching server and training
# runs large-batch steps there; small-batch paths (eval, arena) use CPU replicas.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Outcome convention (AlphaZero control) ────────────────────────────────────
# Standard AlphaZero: the network predicts a POLICY (a distribution over
# actions) and a scalar VALUE in [-1, 1] from the mover's perspective. Chess
# has draws, which map to value 0 exactly (chess's own utility scale:
# WinUtility=+1, DrawUtility=0, LossUtility=-1). The MCTS-Solver still proves
# exact WIN/DRAW/LOSS outcomes; those codes flip one ply up (win<->loss, draw
# stays) as the mover alternates.
_WIN, _DRAW, _LOSS = 0, 1, 2
_TERM_VALUE = np.array([1.0, 0.0, -1.0])           # value of a proven outcome
_FLIP_TERM  = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)


# ── Input helpers ─────────────────────────────────────────────────────────────
# Chess's observation tensor is ALREADY a dense (20, 8, 8) plane stack (12
# piece planes + 1 empty plane + repetition-count + side-to-play +
# irreversible-move-counter + 4 castling-rights planes) — no reshape needed,
# unlike Boop's flat-184-float observation. It is NOT player-relative (White
# and Black pieces occupy fixed planes regardless of who is asking; a
# side-to-play plane tells the network whose turn it is), which is also why
# there is no useful board-symmetry augmentation for chess — see the intro cell.

def state_to_tensor(state, device):
    """Single game state → (1, 20, 8, 8) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = obs.reshape(1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat observations → (B, 20, 8, 8) float tensor. Accepts python
    lists or numpy arrays of any float dtype (replay samples are stored fp16 —
    see the tree-ops cell) and converts in ONE pass."""
    obs = np.asarray(obs_list, dtype=np.float32)
    x   = obs.reshape(-1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


# ── Device capability probe: on-device gather (index_select) ─────────────────
# The two hottest data paths in this notebook both want to GATHER a handful of
# legal-action entries (~35 of 4674) out of the dense (B, 4674, ·) head
# outputs while they are still ON the device, so only ~1% of the data ever
# crosses the bus: (a) the self-play inference server's response rows, and
# (b) the training loss (which only scores evidence entries anyway).
# GPU→CPU readback is DirectML's single most expensive primitive (staging-
# buffer copy + full pipeline sync), so this matters far more there than on
# CUDA. torch-directml's op coverage varies by version, though — so probe
# index_select once here, against known values, forward and backward
# separately, and let each consumer fall back to a full-tensor download when
# its direction is unsupported (correct either way, just slower).
def _probe_device_gather(device):
    if str(device) == 'cpu':
        return True, True                 # plain aten ops — always available
    fwd = bwd = False
    try:
        x   = torch.arange(12.0, device=device).reshape(4, 3)
        idx = torch.tensor([2, 0, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx).cpu()
        fwd = bool(torch.equal(y, torch.tensor([[6., 7., 8.],
                                                [0., 1., 2.],
                                                [9., 10., 11.]])))
    except Exception:
        pass
    try:
        x   = torch.ones(4, 3, device=device, requires_grad=True)
        idx = torch.tensor([1, 1, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx)
        y.backward(torch.ones_like(y))    # backward of gather = index_add
        g   = x.grad.detach().cpu().sum(dim=1)
        bwd = bool(torch.allclose(g, torch.tensor([0., 6., 0., 3.])))
    except Exception:
        pass
    return fwd, bwd

_GATHER_FWD_OK, _GATHER_BWD_OK = _probe_device_gather(DEVICE)
if str(DEVICE) != 'cpu':
    print(f'on-device gather: forward '
          f'{"OK" if _GATHER_FWD_OK else "UNAVAILABLE (full-download fallback)"}'
          f', backward '
          f'{"OK" if _GATHER_BWD_OK else "UNAVAILABLE (full-download fallback)"}')


# ── Network modules (identical to the Boop notebooks) ─────────────────────────

class _GroupNorm(nn.Module):
    """GroupNorm from elementwise ops — DirectML-safe (the fused kernel's
    backward is broken there) and keeps NO running stats (train == eval)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


def _softplus(x):
    """Numerically-stable softplus from elementary ops only. torch's fused
    aten::softplus has no DirectML kernel; relu/abs/exp/log/add all do (softmax
    already exercises exp+log on DML). Identical to F.softplus:
    log(1+e^x) = max(x,0) + log(1 + e^-|x|)."""
    return torch.relu(x) + torch.log(1.0 + torch.exp(-torch.abs(x)))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class AlphaZeroChessNet(nn.Module):
    """
    Standard AlphaZero network for Chess: a POLICY head (logits over all
    actions) + a scalar VALUE head. This is the control against ThompsonZero —
    the body (conv stem + SE-ResBlocks) and every DirectML-safe primitive are
    shared verbatim; only the heads and the training targets differ.

    Input  : (B, 20, 8, 8) — chess's native observation planes
    Body   : Conv stem -> N x ResBlock(channels, SE)      [shared with Thompson]
    Head   : 1x1 conv (H ch) -> flatten ->
               policy_out Linear(H*64, NUM_ACTIONS)  -> logits (softmax over
                          LEGAL actions is applied at use-time, not in the net)
               value_out  Linear(H*64, 64) -> ReLU -> Linear(64, 1) -> tanh
                          -> value in [-1, 1] (mover's perspective)

    forward(x) -> (policy_logits (B, NUM_ACTIONS), value (B,)).
    """
    _HEAD_CH = 8      # matches the Thompson net so parameter counts are close

    def __init__(self, channels=128, num_blocks=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(_OBS_SHAPE[0], channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, self._HEAD_CH, 1, bias=False),
            _norm(self._HEAD_CH),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        flat = self._HEAD_CH * _OBS_SHAPE[1] * _OBS_SHAPE[2]
        self.policy_out = nn.Linear(flat, _NUM_ACTIONS)
        self.value_out  = nn.Sequential(
            nn.Linear(flat, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))

    def forward(self, x):
        h      = self.head(self.body(self.stem(x)))
        logits = self.policy_out(h)                      # (B, NUM_ACTIONS)
        value  = torch.tanh(self.value_out(h)).squeeze(-1)   # (B,) in [-1, 1]
        return logits, value


print(f'Device: {DEVICE}')
_demo = AlphaZeroChessNet()
print(f'AlphaZeroChessNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AlphaZero MCTS: PUCT selection, value backup, MCTS-Solver — the CONTROL
# against ThompsonZero. This is standard AlphaZero: each node keeps per-edge
# VISIT COUNTS N and value sums W (Q = W/N); selection is PUCT
# (Q + c·P·√ΣN/(1+N)); a leaf's scalar NN value is backed up the path, flipping
# sign every ply. The policy target is the root's visit distribution and the
# value target is the game outcome z — nothing Bayesian, no belief
# distributions. What IS shared with the Thompson notebook (verbatim or nearly):
# the chess game, the network body, the DirectML-safe primitives, the sparse
# legal-action wire format, the multiprocess self-play pool, the Elo eval
# harness, checkpointing, and the draw/buffer diagnostics — so a strength
# difference between the two isolates the SEARCH+TARGET design, not the setup.
# ══════════════════════════════════════════════════════════════════════════════
import math
import random
import itertools
from collections import defaultdict

# No board-symmetry augmentation for chess (same reasoning as the Thompson
# notebook: absolute, non-player-relative planes; King/Queen break a mirror).
# A training sample is (obs fp16, action_idx int32 (m,), policy fp32 (m,),
# value fp32 scalar, solved bool): sparse legal-action policy target + a scalar
# value target. `solved` tags a solver-proven position (a diagnostic).

def augment_sample(obs_flat, act_idx, policy, value, solved=False):
    """No-op wrapper (kept for call-site parity with the Boop notebooks):
    returns the single (obs, action-idx, policy, value, solved) sample in a
    list."""
    return [(np.asarray(obs_flat, dtype=np.float16),
             np.asarray(act_idx,  dtype=np.int32),
             np.asarray(policy,   dtype=np.float32),
             float(value), bool(solved))]


C_PUCT   = 1.5        # PUCT exploration constant
_FPU     = 0.0        # first-play-urgency value for unvisited edges (mover view)

_MAX_EVAL_PLIES = None  # set below; forward declaration for linters


def _softmax_legal(logits):
    """Softmax of the legal-action logits → prior over legal moves."""
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()


class _SNode:
    """One expanded state. Per LEGAL edge i: prior P[i], visits N[i], value sum
    W[i] (from THIS node's mover's perspective), virtual loss vloss[i], and a
    proven-outcome marker term[i] (>=0 → _WIN/_DRAW/_LOSS for the mover).
    `obs` is the fp16 observation, stashed for the training sample."""
    __slots__ = ('player', 'legal', 'P', 'N', 'W', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, priors):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        k = len(self.legal)
        self.P        = np.asarray(priors, dtype=np.float64)
        self.N        = np.zeros(k, dtype=np.int64)
        self.W        = np.zeros(k, dtype=np.float64)
        self.vloss    = np.zeros(k, dtype=np.int64)
        self.term     = np.full(k, -1, dtype=np.int8)
        self.children = [None] * k
        self.obs      = None


def _edge_scores(node):
    """PUCT score per edge (from node.player's perspective). Proven edges use
    their exact outcome value so a proven win dominates; virtual loss both
    discounts an in-flight edge's Q and inflates its visit count in U."""
    N, W, vl, t = node.N, node.W, node.vloss, node.term
    ne = N + vl
    Q = np.where(ne > 0, W / np.maximum(ne, 1), _FPU)
    Q = Q - vl * 1.0 / np.maximum(ne, 1)          # extra virtual-loss discount
    if (t >= 0).any():
        Q = Q.copy()
        Q[t >= 0] = _TERM_VALUE[t[t >= 0]]        # exact proven value
    sqrt_sum = math.sqrt(max(1, int(ne.sum())))
    U = C_PUCT * node.P * sqrt_sum / (1.0 + ne)
    return Q + U


def _forced_child(node, k_forced):
    """KataGo forced playouts (Wu 2020): at the (noised) root, every child must
    receive at least sqrt(k·P·N) visits so Dirichlet-seeded moves are actually
    explored; the largest-deficit child is forced. Policy-target pruning later
    removes visits the forced exploration didn't earn. Returns a forced edge
    index, or None to fall through to plain PUCT. Self-play only (k_forced>0)."""
    ne = node.N + node.vloss
    total = int(ne.sum())
    if total > 0:
        need = np.sqrt(k_forced * node.P * total) - ne
        if need.max() > 0.0:
            return int(need.argmax())
    return None


def _select_leaf(root, root_state, rng, k_forced=0.0):
    """Descend by PUCT until an unexpanded or terminal/proven edge. Applies
    virtual loss along the path. `k_forced>0` enables KataGo forced playouts at
    the root (self-play). Returns (path, leaf_state_or_None, edge):
      proven/terminal edge → (path, None, None)   (outcome recorded in term)
      unexpanded edge      → (path, state-at-leaf, (node, idx))  (needs NN)."""
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = None
        if node is root and k_forced > 0.0:
            idx = _forced_child(node, k_forced)
        if idx is None:
            idx = int(_edge_scores(node).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]           # exactly +1 / 0 / -1
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path, leaf_value, leaf_player):
    """Standard AlphaZero value backup: remove virtual loss, increment visits,
    add the leaf value from each ancestor node's own mover's perspective
    (flip when the mover differs from the leaf's mover)."""
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        node.N[idx] += 1
        node.W[idx] += leaf_value if node.player == leaf_player else -leaf_value


# ── MCTS-Solver (identical logic to the Thompson notebook) ────────────────────
def _node_solved_outcome(node):
    """Proven outcome for node.player if solved, else None: WIN as soon as any
    edge is a proven win; else DRAW if any edge proven draw and all edges
    proven; else LOSS only once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _propagate_solved(path, aux=None):
    """Walk leaf→root; whenever a node is fully solved, prove the parent edge
    entering it (flipped). Emits each newly-solved node once into `aux` as a
    solver-labeled training sample (exact policy+value ground truth)."""
    for k in range(len(path) - 1, 0, -1):
        node = path[k][0]
        out = _node_solved_outcome(node)
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]
        if aux is not None and node.obs is not None:
            ai, pol, val = _solved_target(node)
            aux.append((node.obs, ai, pol, val))


def _backup_terminal(path, leaf_value, leaf_player, aux=None):
    """A path that ended on a proven/terminal edge: back up the exact value,
    then propagate any newly-solved nodes (emitting solver samples into aux)."""
    _backup(path, leaf_value, leaf_player)
    _propagate_solved(path, aux)


def _solved_target(node):
    """Exact training target for a SOLVED node: policy = uniform over the
    optimal (proven-best) edges, value = the proven outcome value. For a
    proven win: mass on the proven-win edges. For a proven draw: mass on the
    proven-draw edges. For a proven loss: uniform over all (all lose)."""
    out = _node_solved_outcome(node)
    t = node.term
    if out == _WIN:
        mask = (t == _WIN)
    elif out == _DRAW:
        mask = (t == _DRAW)
    else:
        mask = np.ones(len(t), dtype=bool)
    pol = mask.astype(np.float32)
    pol /= pol.sum()
    return node.legal.copy(), pol, float(_TERM_VALUE[out])


# ── Self-play move selection + targets ────────────────────────────────────────
def _solved_adjust_counts(node, counts):
    """Fold MCTS-Solver certainty into raw visit counts (KataGo-style): a
    proven WIN must dominate move choice and the policy target regardless of
    how few visits it got before the search stopped; proven LOSSES are zeroed
    (never played, never taught) as long as a non-loss alternative exists."""
    t = node.term
    if (t == _WIN).any():
        out = np.zeros_like(counts)
        out[t == _WIN] = 1.0
        return out
    if (t == _LOSS).any() and not (t == _LOSS).all():
        out = counts.copy()
        out[t == _LOSS] = 0.0
        if out.sum() > 0:
            return out
    return counts


def _pruned_counts(root, k_forced):
    """KataGo policy-target pruning (Wu 2020): subtract root-child visits that
    exist only because of forced playouts / Dirichlet noise, keeping the visits
    each move earned on merit (those its PUCT score justifies against the
    most-visited child). Children left with <=1 visit are zeroed; the
    most-visited child always keeps every visit. Vectorized port of the
    kataboop implementation."""
    N = root.N.astype(np.float64)
    total = N.sum()
    best = int(N.argmax())
    if k_forced <= 0.0 or total <= 0 or N[best] <= 0:
        return N
    sqrt_total = math.sqrt(total)
    Q = np.where(N > 0, root.W / np.maximum(N, 1), 0.0)
    puct_best = Q[best] + C_PUCT * root.P[best] * sqrt_total / (1.0 + N[best])
    out = N.copy()
    for i in range(len(N)):
        n = N[i]
        if i == best or n == 0:
            continue
        d = puct_best - Q[i]
        if d <= 0:
            continue                               # beats best's PUCT: earned
        n_min    = C_PUCT * root.P[i] * sqrt_total / d - 1.0
        n_forced = math.sqrt(k_forced * root.P[i] * total)
        keep = min(n, max(int(math.ceil(n_min)), int(n - n_forced), 0))
        out[i] = 0.0 if keep <= 1 else keep
    return out


def root_policy_target(root, k_forced=0.0):
    """Policy training target = the root's visit distribution over legal moves,
    KataGo-PRUNED (k_forced>0 removes forced-playout visits the moves didn't
    earn) and solver-adjusted, as a sparse (action_idx, prob) pair over the
    surviving edges. AlphaZero improvement operator."""
    counts = _pruned_counts(root, k_forced)
    counts = _solved_adjust_counts(root, counts)
    tot = counts.sum()
    if tot <= 0:                                   # no visits (root proven pre-search)
        counts = root.P.copy(); tot = counts.sum()
    keep = counts > 0
    return root.legal[keep].astype(np.int32), (counts[keep] / tot).astype(np.float32)


def root_pick(root, rng, temperature):
    """Choose the move to PLAY: sample ∝ N^(1/temp) (exploration) or, for
    temperature ~0, the most-visited move — with a prior tie-break so all-zero
    or tied counts never degenerate to the first legal action."""
    counts = _solved_adjust_counts(root, root.N.astype(np.float64))
    if temperature < 1e-2 or counts.sum() == 0:
        idx = int(np.argmax(counts + 1e-6 * root.P))   # prior breaks ties
    else:
        logits = np.log(counts + 1e-10) / temperature
        logits -= logits.max()
        p = np.exp(logits); p /= p.sum()
        idx = int(rng.choice(len(counts), p=p))
    return int(root.legal[idx])


def root_value(root):
    """Root Q (mover's perspective), the search's value estimate — used for the
    optional z/root value-target mix and diagnostics."""
    ne = root.N
    tot = ne.sum()
    return float((root.W[ne > 0].sum()) / tot) if tot > 0 else 0.0


def add_dirichlet_noise(root, rng, alpha, frac):
    """Standard AlphaZero root exploration: mix Dirichlet(alpha) noise into the
    root priors (in place)."""
    if frac <= 0.0:
        return
    noise = rng.dirichlet([alpha] * len(root.P))
    root.P = (1.0 - frac) * root.P + frac * noise


def make_root(state, legal, logits, rng, dirichlet_alpha=0.0, dirichlet_frac=0.0):
    root = _SNode(state.current_player(), legal, _softmax_legal(logits))
    if dirichlet_frac > 0.0:
        add_dirichlet_noise(root, rng, dirichlet_alpha, dirichlet_frac)
    return root


# ── Batched-leaf AlphaZero bot (single-process; used by eval + tests) ─────────
class AZMCTSBot:
    """mcts_search(state) → root _SNode after `max_simulations` PUCT
    simulations, collecting `batch_size` leaves per wave (virtual loss keeps a
    wave diverse) for one NN forward pass. Root Dirichlet noise off by default
    (evaluation is noise-free)."""

    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, random_state=None,
                 dirichlet_alpha=0.0, dirichlet_frac=0.0):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self._rng            = random_state or np.random.RandomState()
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_frac  = dirichlet_frac

    def _eval_batch(self, states):
        obs = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs, self.device)
        with torch.no_grad():
            logits, values = self.network(x)
        return logits.cpu().numpy(), values.cpu().numpy()

    def mcts_search(self, state):
        logits, values = self._eval_batch([state])
        legal = state.legal_actions()
        root = make_root(state, legal, logits[0][legal], self._rng,
                         self.dirichlet_alpha, self.dirichlet_frac)
        sims = 0
        while sims < self.max_simulations:
            if _node_solved_outcome(root) is not None:
                break
            wave    = min(self.batch_size, self.max_simulations - sims)
            pending = []
            for _ in range(wave):
                path, st, edge = _select_leaf(root, state, self._rng)
                if st is None:                     # terminal / proven
                    node, idx = path[-1]
                    v = _TERM_VALUE[node.term[idx]]
                    _backup_terminal(path, v, node.player)
                    sims += 1
                else:
                    pending.append((path, st, edge))
            unique = {}
            for path, st, (node, idx) in pending:
                unique.setdefault((id(node), idx), (node, idx, st))
            if unique:
                entries = list(unique.values())
                lg, vl = self._eval_batch([st for _, _, st in entries])
                made = {}
                for (node, idx, st), l_row, v_row in zip(entries, lg, vl):
                    leg = st.legal_actions()
                    ch = _SNode(st.current_player(), leg,
                                _softmax_legal(l_row[leg]))
                    node.children[idx] = ch
                    made[(id(node), idx)] = float(v_row)
            for path, st, (node, idx) in pending:
                v = made[(id(node), idx)]
                _backup(path, v, node.children[idx].player)
                sims += 1
        return root


def make_az_bot(game, network, device, num_simulations, batch_size=16,
                dirichlet_alpha=0.0, dirichlet_frac=0.0):
    return AZMCTSBot(game, network, device, num_simulations, batch_size=batch_size,
                     dirichlet_alpha=dirichlet_alpha, dirichlet_frac=dirichlet_frac)


# ── Evaluation ────────────────────────────────────────────────────────────────
def policy_action(network, state, device, sample=False, rng=None):
    """Search-free move from the raw policy head: argmax over legal logits, or
    (sample=True) a draw from the legal-softmax policy."""
    with torch.no_grad():
        logits, _ = network(state_to_tensor(state, device))
    logits = logits[0].cpu().numpy()
    legal = state.legal_actions()
    p = _softmax_legal(logits[legal])
    if sample:
        rng = rng or np.random
        return int(legal[rng.choice(len(legal), p=p)])
    return int(legal[int(p.argmax())])


# Evaluation-side ply cap (identical rationale to the Thompson notebook: chess
# is only eventually bounded by the 50-move rule; a weak net can run very long
# and stall a serial tournament). A cut-off game scores as a draw — exact,
# since returns() on a non-terminal chess state is [0.0, 0.0].
MAX_EVAL_PLIES = 300


def play_match(net_a, net_b, game, n_games, device):
    """net_a vs net_b, alternating colours, search-free sampled moves. net ==
    None → random. Returns (wins_a, draws, wins_b) — the quick progress pulse."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = policy_action(net, state, device, sample=True)
            state.apply_action(action)
            ply += 1
        if not state.is_terminal():
            dd += 1
            continue
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Most-visited move after a full search (noise-free), with the prior
    tie-break so a search that visited nothing never plays the first legal
    action."""
    root = bot.mcts_search(state)
    counts = _solved_adjust_counts(root, root.N.astype(np.float64))
    idx = int(np.argmax(counts + 1e-6 * root.P))
    return int(root.legal[idx])


# ── Unified eval pool: one shared Elo table over (checkpoint × sim-budget) ────
# Every checkpoint enters the pool as MULTIPLE players — one per eval sim budget
# (e.g. label@0 = search-free policy head, label@32, label@128) — plus a single
# 'random' mover. All share ONE Elo table, so search-vs-no-search and
# generation-vs-generation are directly comparable in the same ratings. A
# player is a dict {key, label, sims, net}. Structure is identical to the
# ThompsonZero notebook's pool (AZ-MCTS bot in place of the Thompson bot), so
# the two experiments' eval numbers are directly comparable.

def _eval_pick(player, state, device, bots):
    net = player['net']
    if net is None:
        return random.choice(state.legal_actions())
    if player['sims'] <= 0:
        return policy_action(net, state, device, sample=False)
    return _mcts_move(bots[player['key']], state)


def _play_eval_game(pa, pb, game, device, bots, elos, wdl, k, opening_plies):
    """One game pa (player 0) vs pb (player 1); update Elo + W/D/L in place."""
    state = game.new_initial_state()
    ply = 0
    while not state.is_terminal() and ply < MAX_EVAL_PLIES:
        if ply < opening_plies:
            action = random.choice(state.legal_actions())
        else:
            action = _eval_pick(pa if state.current_player() == 0 else pb,
                                state, device, bots)
        state.apply_action(action)
        ply += 1
    r  = state.returns()[0]
    s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
    ka, kb = pa['key'], pb['key']
    e0 = 1.0 / (1.0 + 10 ** ((elos[kb] - elos[ka]) / 400.0))
    elos[ka] += k * (s0 - e0)
    elos[kb] += k * ((1.0 - s0) - (1.0 - e0))
    cell = wdl[(ka, kb)]
    if r > 0:   cell[0] += 1
    elif r < 0: cell[2] += 1
    else:       cell[1] += 1


def run_eval_pool(players, elos, wdl, game, device, new_keys,
                  k=32.0, games_per_pair=2, random_mult=5, opening_plies=0):
    """Rate the pool. Each of the `new_keys` players plays `games_per_pair`
    games (alternating colours) against every player that ALREADY existed
    (new-vs-new is left to the random phase); then `random_mult * len(players)`
    extra games are played between uniformly-random distinct pairs over ALL
    players, to keep older ratings mixing. `elos` must hold a rating for every
    player key. Bots for search players are built once and reused."""
    keys = [p['key'] for p in players]
    by_key = {p['key']: p for p in players}
    bots = {}
    for p in players:
        if p['net'] is not None and p['sims'] > 0:
            bots[p['key']] = make_az_bot(
                game, p['net'], device, p['sims'],
                batch_size=max(1, min(p['sims'], 16)))
    new = set(new_keys)
    existing = [key for key in keys if key not in new]
    schedule = []
    half = games_per_pair // 2
    for nk in new_keys:
        for ek in existing:
            schedule += [(nk, ek)] * half + [(ek, nk)] * half
            schedule += [(nk, ek)] * (games_per_pair - 2 * half)
    if len(keys) >= 2:
        for _ in range(random_mult * len(keys)):
            a, b = random.sample(keys, 2)
            schedule.append((a, b))
    random.shuffle(schedule)
    for ka, kb in schedule:
        _play_eval_game(by_key[ka], by_key[kb], game, device, bots,
                        elos, wdl, k, opening_plies)

In [ ]:
# ── Parallel AlphaZero self-play: many games, one forward pass ────────────────
# Same orchestration as the Thompson notebook (interleave N games' leaf waves
# into single NN passes; opponent-pool diversity incl. a random mover; draw /
# buffer diagnostics), on the PUCT tree above. The value target is the game
# outcome z, resolved at game end (mixed with the root search value by
# VALUE_LAMBDA); the policy target is the root visit distribution.
import os


def _nn_eval_states(network, device, states):
    """States → (policy_logits (B, A), value (B,), obs fp16 (B, 1280))."""
    obs16 = np.asarray([s.observation_tensor(s.current_player())
                        for s in states], dtype=np.float16)
    x = batch_to_tensor(obs16, device)
    with torch.no_grad():
        logits, values = network(x)
    return logits.cpu().numpy(), values.cpu().numpy(), obs16


def _resolve_values(hist, z_player, terminal, value_lambda):
    """Turn per-move records (obs, ai, pol, root_v, player) into finished
    (obs, ai, pol, value) samples. Terminal games mix the outcome
    z with the root search value (value_lambda·z + (1−λ)·root_v); cut-off games
    have no observed outcome, so they keep the pure search value (bootstrap)."""
    out = []
    for obs, ai, pol, root_v, player in hist:
        if terminal:
            z = z_player[player]
            value = value_lambda * z + (1.0 - value_lambda) * root_v
        else:
            value = root_v
        out.append((obs, ai, pol, float(value)))
    return out


class AZParallelSelfPlay:
    """Interleaves n_parallel PUCT self-play games so their leaf waves share one
    NN forward pass. Weights are read live from `network`."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=8,
                 fast_sims=200, full_sims=800, fast_prob=0.75, temp_threshold=20,
                 value_lambda=1.0, dirichlet_alpha=0.3, dirichlet_frac=0.25,
                 k_forced=2.0, seed=None, pool_prob=0.0, checkpoint_dir=None,
                 channels=None, num_blocks=None, max_selfplay_plies=400,
                 random_pool_frac=0.5):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        self.value_lambda = value_lambda
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_frac = dirichlet_frac
        self.k_forced = k_forced
        self.max_selfplay_plies = max_selfplay_plies
        self.random_pool_frac = random_pool_frac
        self._rng = np.random.RandomState(seed)
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}
        self.last_aux = 0
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = AlphaZeroChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        sims = (self.fast_sims if self._rng.rand() < self.fast_prob
                else self.full_sims)
        rng = self._rng
        slot = {'state': self.game.new_initial_state(), 'hist': [], 'aux': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        if self.pool_prob > 0 and rng.rand() < self.pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if self.checkpoint_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < self.random_pool_frac:
                slot['pool'] = {'label': 'random',
                                'side': int(rng.randint(0, 2)), 'net': None}
            else:
                label = labels[rng.randint(len(labels))]
                slot['pool'] = {'label': label,
                                'side': int(rng.randint(0, 2)),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            legal = state.legal_actions()
            if pool['label'] == 'random':
                action = int(legal[self._rng.randint(len(legal))])
            else:
                logits, _v, _o = _nn_eval_states(pool['net'], 'cpu', [state])
                p = _softmax_legal(logits[0][legal])
                action = int(legal[int(p.argmax())])
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _finish_slot(self, i):
        s = self.slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist3 = _resolve_values(s['hist'], ret, True, self.value_lambda)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
        else:
            hist3 = _resolve_values(s['hist'], None, False, self.value_lambda)
            result = 'cutoff'
        self.last_aux = len(s['aux'])
        self.stats['games'] += 1
        self.stats[result] += 1
        self.stats['plies'] += int(s['move'])
        self.slots[i] = self._new_game()
        return hist3 + s['aux']

    def episodes(self):
        while True:
            for data in self._step():
                yield data

    def _step(self):
        rng     = self._rng
        done    = self._resolve_pool_moves()
        pending = []
        evals   = []
        seen    = set()
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue
            if s['root'] is None:
                evals.append(('root', i, None, s['state']))
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue
            wave = min(self.wave, s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], s['state'], rng,
                                              self.k_forced)
                if st is None:
                    node, idx = path[-1]
                    v = _TERM_VALUE[node.term[idx]]
                    _backup_terminal(path, v, node.player, s['aux'])
                    s['n'] += 1
                else:
                    node, idx = edge
                    pending.append((i, path, edge))
                    if (id(node), idx) not in seen:
                        seen.add((id(node), idx))
                        evals.append(('leaf', node, idx, st))

        made = {}
        if evals:
            logits, values, obs16 = _nn_eval_states(
                self.network, self.device, [e[3] for e in evals])
            for (kind, a, b, st), l_row, v_row, o_row in zip(evals, logits,
                                                             values, obs16):
                leg = st.legal_actions()
                node = _SNode(st.current_player(), leg, _softmax_legal(l_row[leg]))
                node.obs = o_row
                if kind == 'root':
                    if self.dirichlet_frac > 0.0:
                        add_dirichlet_noise(node, rng, self.dirichlet_alpha,
                                            self.dirichlet_frac)
                    self.slots[a]['root'] = node
                else:
                    a.children[b] = node
                    made[(id(a), b)] = float(v_row)
        for i, path, (node, idx) in pending:
            v = made[(id(node), idx)]
            _backup(path, v, node.children[idx].player)
            self.slots[i]['n'] += 1

        for i, s in enumerate(self.slots):
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(self._finish_slot(i))
        return done

    def _play_move(self, i):
        s     = self.slots[i]
        state = s['state']
        root  = s['root']
        obs   = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
        ai, pol = root_policy_target(root, self.k_forced)
        temp = 1.0 if s['move'] < self.temp_threshold else 0.0
        # Per-move record: value filled at game end (needs the outcome z).
        s['hist'].append((obs, ai, pol, root_value(root), root.player))
        action = root_pick(root, self._rng, temp)
        state.apply_action(action)
        s['move'] += 1
        s['root'] = None
        s['n'] = 0

In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# Same design as the Boop notebooks: N CPU worker processes do ALL game and
# tree work (pure Python — processes sidestep the GIL) and ship their NN
# requests to ONE server thread here, which batches requests from all workers
# into single large forward passes on the GPU. Small-batch inference (eval
# games, tournaments, arena) runs on CPU replicas elsewhere.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by chess_alphazero_training.ipynb — do not edit by hand.
# Self-play worker module: AlphaZero PUCT tree ops + worker loop. Loads chess
# directly (pyspiel.load_game('chess')) — no custom engine to embed, unlike
# the Boop notebooks. Exists as a real file because Windows multiprocessing
# (spawn) cannot pickle functions defined inside notebook cells.

# ═══════════════════════════════════════════════════════════════════════════════
# AlphaZero MCTS tree ops — standalone copy of the notebook's PUCT semantics for
# the self-play workers (pure CPU/numpy, no torch; every NN eval is shipped to
# the main process's GPU inference server). Loads chess directly. Mirrors the
# tree-ops cell exactly; see there for the rationale.
# ═══════════════════════════════════════════════════════════════════════════════
import os
import math
import numpy as np
import pyspiel

_GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = _GAME.num_distinct_actions()

_WIN, _DRAW, _LOSS = 0, 1, 2
_TERM_VALUE = np.array([1.0, 0.0, -1.0])
_FLIP_TERM  = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)

C_PUCT = 1.5
_FPU   = 0.0


def _softmax_legal(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()


class _SNode:
    __slots__ = ('player', 'legal', 'P', 'N', 'W', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, priors):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        k = len(self.legal)
        self.P        = np.asarray(priors, dtype=np.float64)
        self.N        = np.zeros(k, dtype=np.int64)
        self.W        = np.zeros(k, dtype=np.float64)
        self.vloss    = np.zeros(k, dtype=np.int64)
        self.term     = np.full(k, -1, dtype=np.int8)
        self.children = [None] * k
        self.obs      = None


def _edge_scores(node):
    N, W, vl, t = node.N, node.W, node.vloss, node.term
    ne = N + vl
    Q = np.where(ne > 0, W / np.maximum(ne, 1), _FPU)
    Q = Q - vl * 1.0 / np.maximum(ne, 1)
    if (t >= 0).any():
        Q = Q.copy()
        Q[t >= 0] = _TERM_VALUE[t[t >= 0]]
    sqrt_sum = math.sqrt(max(1, int(ne.sum())))
    U = C_PUCT * node.P * sqrt_sum / (1.0 + ne)
    return Q + U


def _forced_child(node, k_forced):
    # KataGo forced playouts at the root (self-play): force the largest-deficit
    # child so noised moves get sqrt(k·P·N) visits. See the tree-ops cell.
    ne = node.N + node.vloss
    total = int(ne.sum())
    if total > 0:
        need = np.sqrt(k_forced * node.P * total) - ne
        if need.max() > 0.0:
            return int(need.argmax())
    return None


def _select_leaf(root, root_state, rng, k_forced=0.0):
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = None
        if node is root and k_forced > 0.0:
            idx = _forced_child(node, k_forced)
        if idx is None:
            idx = int(_edge_scores(node).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path, leaf_value, leaf_player):
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        node.N[idx] += 1
        node.W[idx] += leaf_value if node.player == leaf_player else -leaf_value


def _node_solved_outcome(node):
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _solved_target(node):
    out = _node_solved_outcome(node)
    t = node.term
    if out == _WIN:
        mask = (t == _WIN)
    elif out == _DRAW:
        mask = (t == _DRAW)
    else:
        mask = np.ones(len(t), dtype=bool)
    pol = mask.astype(np.float32)
    pol /= pol.sum()
    return node.legal.copy(), pol, float(_TERM_VALUE[out])


def _propagate_solved(path, aux=None):
    for k in range(len(path) - 1, 0, -1):
        node = path[k][0]
        out = _node_solved_outcome(node)
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]
        if aux is not None and node.obs is not None:
            ai, pol, val = _solved_target(node)
            aux.append((node.obs, ai, pol, val))


def _backup_terminal(path, leaf_value, leaf_player, aux=None):
    _backup(path, leaf_value, leaf_player)
    _propagate_solved(path, aux)


def _solved_adjust_counts(node, counts):
    t = node.term
    if (t == _WIN).any():
        out = np.zeros_like(counts)
        out[t == _WIN] = 1.0
        return out
    if (t == _LOSS).any() and not (t == _LOSS).all():
        out = counts.copy()
        out[t == _LOSS] = 0.0
        if out.sum() > 0:
            return out
    return counts


def _pruned_counts(root, k_forced):
    # KataGo policy-target pruning (Wu 2020) — see the tree-ops cell.
    N = root.N.astype(np.float64)
    total = N.sum()
    best = int(N.argmax())
    if k_forced <= 0.0 or total <= 0 or N[best] <= 0:
        return N
    sqrt_total = math.sqrt(total)
    Q = np.where(N > 0, root.W / np.maximum(N, 1), 0.0)
    puct_best = Q[best] + C_PUCT * root.P[best] * sqrt_total / (1.0 + N[best])
    out = N.copy()
    for i in range(len(N)):
        n = N[i]
        if i == best or n == 0:
            continue
        d = puct_best - Q[i]
        if d <= 0:
            continue
        n_min    = C_PUCT * root.P[i] * sqrt_total / d - 1.0
        n_forced = math.sqrt(k_forced * root.P[i] * total)
        keep = min(n, max(int(math.ceil(n_min)), int(n - n_forced), 0))
        out[i] = 0.0 if keep <= 1 else keep
    return out


def root_policy_target(root, k_forced=0.0):
    counts = _pruned_counts(root, k_forced)
    counts = _solved_adjust_counts(root, counts)
    tot = counts.sum()
    if tot <= 0:
        counts = root.P.copy(); tot = counts.sum()
    keep = counts > 0
    return root.legal[keep].astype(np.int32), (counts[keep] / tot).astype(np.float32)


def root_pick(root, rng, temperature):
    counts = _solved_adjust_counts(root, root.N.astype(np.float64))
    if temperature < 1e-2 or counts.sum() == 0:
        idx = int(np.argmax(counts + 1e-6 * root.P))
    else:
        logits = np.log(counts + 1e-10) / temperature
        logits -= logits.max()
        p = np.exp(logits); p /= p.sum()
        idx = int(rng.choice(len(counts), p=p))
    return int(root.legal[idx])


def root_value(root):
    ne = root.N
    tot = ne.sum()
    return float((root.W[ne > 0].sum()) / tot) if tot > 0 else 0.0


def add_dirichlet_noise(root, rng, alpha, frac):
    if frac <= 0.0:
        return
    noise = rng.dirichlet([alpha] * len(root.P))
    root.P = (1.0 - frac) * root.P + frac * noise


def _resolve_values(hist, z_player, terminal, value_lambda):
    out = []
    for obs, ai, pol, root_v, player in hist:
        if terminal:
            value = value_lambda * z_player[player] + (1.0 - value_lambda) * root_v
        else:
            value = root_v
        out.append((obs, ai, pol, float(value)))
    return out


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined AlphaZero self-play worker (leapfrog halves, same protocol as
    the Thompson worker). NN requests are tagged 'live' or a benchmark label;
    'random' opponents play locally with no request. Response per row is
    (logits_k (k,) fp32, value scalar) — the server gathers the legal policy
    logits and the scalar value from the dense head outputs."""
    rng  = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    game = _GAME
    pool_prob   = cfg.get('pool_prob', 0.0)
    pool_dir    = cfg.get('checkpoint_dir')
    max_plies   = cfg.get('max_selfplay_plies', 400)
    value_lambda = cfg.get('value_lambda', 1.0)
    temp_thresh = cfg['temp_threshold']
    dir_alpha   = cfg.get('dirichlet_alpha', 0.3)
    dir_frac    = cfg.get('dirichlet_frac', 0.25)
    k_forced    = cfg.get('k_forced', 2.0)         # KataGo forced playouts
    rand_pool_frac = cfg.get('random_pool_frac', 0.5)

    def new_game():
        sims = (cfg['fast_sims'] if rng.rand() < cfg['fast_prob']
                else cfg['full_sims'])
        slot = {'state': game.new_initial_state(), 'hist': [], 'aux': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        if pool_prob > 0 and rng.rand() < pool_prob:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')] \
                    if pool_dir else []
            except OSError:
                labels = []
            if not labels or rng.rand() < rand_pool_frac:
                label = 'random'
            else:
                label = labels[rng.randint(len(labels))]
            slot['pool'] = {'label': label, 'side': int(rng.randint(0, 2))}
        return slot

    def finish_and_reset(i):
        s = slots[i]
        st = s['state']
        term = st.is_terminal()
        if term:
            ret = st.returns()
            hist3 = _resolve_values(s['hist'], ret, True, value_lambda)
            result = 'draw' if (ret[0] == 0.0 and ret[1] == 0.0) else 'decisive'
        else:
            hist3 = _resolve_values(s['hist'], None, False, value_lambda)
            result = 'cutoff'
        episode_q.put((hist3 + s['aux'], len(s['aux']), result, int(s['move'])))
        slots[i] = new_game()

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        evals, paths, obs, legals = [], [], [], []
        seen = set()
        for i in idxs:
            s = slots[i]
            st0 = s['state']
            pool = s['pool']
            if pool is not None and st0.current_player() == pool['side']:
                continue
            if s['root'] is None:
                leg = st0.legal_actions()
                o = np.asarray(st0.observation_tensor(st0.current_player()),
                               dtype=np.float16)
                evals.append(('root', i, leg, o))
                obs.append(o)
                legals.append(np.asarray(leg, dtype=np.int32))
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue
            wave = min(cfg['wave'], s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], st0, rng, k_forced)
                if st is None:
                    node, idx = path[-1]
                    _backup_terminal(path, _TERM_VALUE[node.term[idx]],
                                     node.player, s['aux'])
                    s['n'] += 1
                    continue
                node, idx = edge
                paths.append((i, path, node, idx))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    leg = st.legal_actions()
                    o = np.asarray(st.observation_tensor(st.current_player()),
                                   dtype=np.float16)
                    evals.append(('leaf', node, idx, st.current_player(), leg, o))
                    obs.append(o)
                    legals.append(np.asarray(leg, dtype=np.int32))
        return evals, paths, obs, legals

    def apply_and_advance(idxs, evals, paths, resp):
        made = {}
        if evals:
            for e, (logits_k, value) in zip(evals, resp):
                if e[0] == 'root':
                    _, i, leg, o = e
                    st = slots[i]['state']
                    nd = _SNode(st.current_player(), leg, _softmax_legal(logits_k))
                    nd.obs = o
                    if dir_frac > 0.0:
                        add_dirichlet_noise(nd, rng, dir_alpha, dir_frac)
                    slots[i]['root'] = nd
                else:
                    _, node, idx, player, leg, o = e
                    nd = _SNode(player, leg, _softmax_legal(logits_k))
                    nd.obs = o
                    node.children[idx] = nd
                    made[(id(node), idx)] = float(value)
        for i, path, node, idx in paths:
            _backup(path, made[(id(node), idx)], node.children[idx].player)
            slots[i]['n'] += 1
        for i in idxs:
            s = slots[i]
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            state = s['state']
            root = s['root']
            o = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
            ai, pol = root_policy_target(root, k_forced)
            temp = 1.0 if s['move'] < temp_thresh else 0.0
            s['hist'].append((o, ai, pol, root_value(root), root.player))
            action = root_pick(root, rng, temp)
            state.apply_action(action)
            s['move'] += 1
            s['root'] = None
            s['n'] = 0
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    def resolve_pool_moves(idxs):
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            legal = state.legal_actions()
            if pool['label'] == 'random':
                action = int(legal[rng.randint(len(legal))])
            else:
                obs = np.asarray(
                    state.observation_tensor(state.current_player()),
                    dtype=np.float16)
                req_q.put((worker_id, pool['label'], obs[None],
                           [np.asarray(legal, dtype=np.int32)]))
                (logits_k, _v), = pool_resp_q.get()
                p = _softmax_legal(logits_k)
                action = int(legal[int(p.argmax())])
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= max_plies:
                finish_and_reset(i)

    inflight = [None, None]
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                evals, paths, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], evals, paths, resp)
                inflight[h] = None
            resolve_pool_moves(halves[h])
            evals, paths, obs, legals = collect(halves[h])
            sent = False
            if evals:
                req_q.put((worker_id, 'live', np.stack(obs), legals))
                sent = True
            inflight[h] = (evals, paths, sent)
'''

pathlib.Path('chess_az_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play AlphaZero self-play; an inference-server
    thread here batches ALL their NN requests into single forward passes on
    `device`. Identical architecture to the Thompson notebook — only the
    response payload differs (gathered legal POLICY LOGITS + a scalar VALUE per
    row, instead of Dirichlet probs+conf). `lock` serializes model access
    between the server thread and training."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Warm the autograd engine's per-device state from the MAIN thread first
        # (as in the Thompson pool: avoids a torch-directml device-ready race).
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import chess_az_worker
        self.procs = [ctx.Process(target=chess_az_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = AlphaZeroChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _forward_gathered(self, net, net_device, xin, flat):
        """One forward pass + on-device gather of the legal POLICY LOGITS
        (~1% of the dense (B, A) head); the scalar VALUE is (B,) so it needs no
        gather. GPU→CPU readback is DirectML's most expensive primitive, so only
        the gathered logits + the tiny value vector cross the bus. Falls back to
        a full download + numpy gather where index_select's forward is
        unsupported (probed in the net cell)."""
        x = torch.from_numpy(xin).to(net_device)
        logits, value = net(x)
        if _GATHER_FWD_OK and str(net_device) != 'cpu':
            ft = torch.from_numpy(flat).to(net_device)
            l_sel = logits.reshape(-1).index_select(0, ft).cpu().numpy()
        else:
            l_sel = logits.reshape(-1).cpu().numpy()[flat]
        v_all = value.reshape(-1).cpu().numpy()
        return l_sel, v_all

    def _serve(self):
        A = _NUM_ACTIONS
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            rows = reqs[0][2].shape[0]
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and rows < 1024:
                try:
                    r = self.req_q.get_nowait()
                    reqs.append(r)
                    rows += r[2].shape[0]
                except _queue.Empty:
                    _time.sleep(0.0003)
            groups = {}
            for wid, net_id, obs, legals in reqs:
                groups.setdefault(net_id, []).append((wid, obs, legals))
            for net_id, group in groups.items():
                net, net_device, needs_lock = self._get_net(net_id)
                obs = np.concatenate([o for _, o, _ in group], axis=0)
                xin = obs.reshape(-1, *_OBS_SHAPE).astype(np.float32)
                row_legals = [l for _, _, ls in group for l in ls]
                flat = np.concatenate([l.astype(np.int64) + r * A
                                       for r, l in enumerate(row_legals)])
                offs = np.zeros(len(row_legals) + 1, dtype=np.int64)
                np.cumsum([len(l) for l in row_legals], out=offs[1:])
                if needs_lock:
                    with self.lock, torch.no_grad():
                        l_sel, v_all = self._forward_gathered(net, net_device,
                                                              xin, flat)
                else:
                    with torch.no_grad():
                        l_sel, v_all = self._forward_gathered(net, net_device,
                                                              xin, flat)
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                ri = 0
                for wid, o, ls in group:
                    out = []
                    for _ in ls:
                        a, b = offs[ri], offs[ri + 1]
                        out.append((l_sel[a:b], float(v_all[ri])))
                        ri += 1
                    target_qs[wid].put(out)

    def episodes(self):
        # Workers ship (samples, n_aux, result, plies). aux/result/plies feed
        # the diagnostics the training loop diffs each eval window.
        self.last_aux = 0
        self.stats = {'games': 0, 'draw': 0, 'cutoff': 0, 'decisive': 0,
                      'plies': 0}
        while True:
            samples, n_aux, result, plies = self.episode_q.get()
            self.last_aux = n_aux
            self.stats['games'] += 1
            self.stats[result] = self.stats.get(result, 0) + 1
            self.stats['plies'] += plies
            yield samples

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in ([self.req_q, self.episode_q] + self.resp_qs
                  + self.pool_resp_qs):
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass

In [ ]:
# ── Hyperparameters (AlphaZero + KataGo optimizations) ────────────────────────
# This is the CONTROL against ThompsonZero: standard AlphaZero (PUCT, visit-
# count policy targets, value = game outcome) PLUS the KataGo optimizations
# proven on Boop in the kata_update branch — forced playouts, policy-target
# pruning, value-target bootstrap, playout-cap randomization, SE-ResBlocks,
# MCTS-Solver. Same chess game, network body, multiprocess pool, eval harness,
# checkpointing, and diagnostics as the Thompson notebook, so a strength
# difference isolates the search+target design, not the hardware or setup.
NUM_EPISODES     = 50_000  # run budget only (LR schedule is decoupled)
# Sim budget cut for FAST FEEDBACK: measured self-play is forward-bound (the
# chess NN forward is ~2.8x Boop's at identical channels/blocks — 8x8 vs 6x6
# board + a 43x bigger policy head), so halving avg sims/move roughly doubles
# games/hour. Early chess AZ is game-starved, not depth-starved: a weak net
# gains more from seeing more positions than from searching each one 1000-deep.
# Raise these again once strength plateaus at this budget.
FULL_MCTS_SIMS   = 400     # was 1000 — high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 100     # was 250  — fast self-play (75%), playout-cap randomization
FAST_GAME_PROB   = 0.75
MAX_SELFPLAY_PLIES = 400   # pacing safety valve; a cut-off game keeps its
                           # search-value targets (no observed outcome — see
                           # cell 6), so nothing recorded is invalidated.
# ── AlphaZero search + target knobs ──────────────────────────────────────────
VALUE_LAMBDA     = 0.5     # value target = λ·z + (1−λ)·root_MCTS_value (KataGo
                           # bootstrap: the final z alone is near coin-flip
                           # noise on a balanced game). 1.0 = pure AlphaZero z.
DIRICHLET_ALPHA  = 0.3     # root exploration noise concentration
DIRICHLET_FRAC   = 0.25    # root prior/noise mix
K_FORCED         = 2.0     # KataGo forced playouts (0 disables); the policy
                           # target is pruned to remove unearned forced visits.
VALUE_LOSS_WEIGHT = 1.0    # value MSE weight relative to policy cross-entropy
TEMP_THRESHOLD   = 20      # visit-count sampled moves per game; argmax after
# ── Hardware parallelism (AMD GPU via DirectML) ──────────────────────────────
import os as _os_hw
USE_WORKERS      = True
SELFPLAY_WORKERS = min(8, max(2, (_os_hw.cpu_count() or 8) - 2))
GAMES_PER_WORKER = 32      # two pipelined halves of 16 → big joint NN batches
WORKER_WAVE      = 8       # leaves/game/wave
N_PARALLEL_GAMES = 8       # single-process fallback (USE_WORKERS=False)
WAVE_PER_GAME    = 8
SELFPLAY_POOL_PROB = 0.15  # fraction of games with one side a frozen benchmark
RANDOM_POOL_FRAC = 0.5     # ...of which this fraction use a uniform-RANDOM
                           # mover (guarantees decisive games early on)
EVAL_EVERY       = 100     # QUICK eval vs the most recent benchmark
                           # (search-free, ~75% draws at this strength — mostly
                           # noise; keeping its frequency low is free wall-clock)
DEEP_EVAL_EVERY  = 1000    # DEEP eval-pool + new benchmark. Cost scales as
                           # ~(6+EVAL_RANDOM_MULT)·N games/eval and the search
                           # players dominate it, so keep this coarse.
QUICK_EVAL_GAMES = 20
# ── Unified eval pool: ONE shared Elo table over (checkpoint × sim-budget) ────
# Identical scheme to the ThompsonZero notebook so the two are directly
# comparable. Every checkpoint enters the SAME table as three players — @0
# (search-free policy argmax), @32, @128 — plus one 'random' mover. When a
# checkpoint is added, its new players each play EVAL_GAMES_PER_PAIR games
# (alternating colours) vs every existing player; then EVAL_RANDOM_MULT·N
# random-pair games run over all players to keep old ratings mixing. The key
# diagnostic is the SEARCH BENEFIT: Elo(@128) − Elo(@0) over training.
EVAL_SIMS        = [0, 32, 128]  # 0 = policy head (no search); 32/128 = AZ-MCTS
EVAL_GAMES_PER_PAIR = 2     # new-vs-existing games (one White, one Black)
EVAL_RANDOM_MULT = 5        # random-pair refresh games = this × (#players)
EVAL_OPENING_PLIES = 4
START_ELO        = 1000.0
K_FACTOR         = 32.0
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8
MAX_BUFFER       = 100_000
CHANNELS         = 128
NUM_BLOCKS       = 10      # SAME body as the Thompson net → identical body
                           # FLOPs, the fair hardware control. NOT reduced for
                           # the fast-feedback pass below: unlike the sim
                           # budget and eval knobs, CHANNELS/NUM_BLOCKS are
                           # baked into the checkpoint's state dict — changing
                           # them would break resume from an existing run
                           # (shape mismatch) rather than just slow it down.
                           # Shrink these ONLY when starting a fresh
                           # CHECKPOINT_DIR, not to speed up a run in progress.
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100
LR_DECAY_EPS     = 30_000
WEIGHT_DECAY     = 1e-4
LR_MIN_FACTOR    = 0.10
CHECKPOINT_DIR   = 'chess_checkpoints_alphazero'
RESUME_FROM_CHECKPOINT = True

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = GAME
base_network = AlphaZeroChessNet(CHANNELS, NUM_BLOCKS).to(DEVICE)
network      = base_network

# torch.compile: same policy as the Thompson notebook (skipped on DirectML /
# when Inductor has no C++ backend; falls back to eager on lazy failure).
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():
                _compiled(torch.zeros(1, *_OBS_SHAPE, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')


class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW built from mul_/add_/addcmul_/addcdiv_ only — DirectML lacks
    aten::lerp. Identical math to torch.optim.AdamW."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    return float(max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)


# ── Loss ──────────────────────────────────────────────────────────────────────
# Standard AlphaZero: policy cross-entropy (over the LEGAL actions — illegal
# logits masked to -inf before the softmax, so the target renormalizes over
# legal moves) + value mean-squared error. All ops have DirectML kernels
# (log_softmax, mul, sum, mse) — unlike the Thompson notebook's lgamma losses,
# no CPU split-backward is needed. Targets are built dense per batch: a legal
# mask, a policy distribution, and a scalar value, all sparse-scattered on CPU
# and uploaded once per step (the dominant cost is the shared conv body, not
# this (B, 4674) head loss).

def policy_value_loss(logits, value, legal_mask, pol_target, val_target):
    """logits (B,A), value (B,); legal_mask (B,A) in {0,1}; pol_target (B,A)
    (rows sum to 1 over legal); val_target (B,). Returns (total, policy CE,
    value MSE)."""
    masked = logits + (legal_mask - 1.0) * 1e9        # illegal → −1e9 (DML-safe)
    logp   = torch.log_softmax(masked, dim=1)
    pol_loss = -(pol_target * logp).sum(dim=1).mean()  # 0·(−1e9)=0 on illegals
    val_loss = ((value - val_target) ** 2).mean()
    return pol_loss + VALUE_LOSS_WEIGHT * val_loss, pol_loss, val_loss


# Self-play source: worker pool (multiprocess + GPU server) or in-process.
import os
import threading
try:
    self_play.shutdown()
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)),
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 pool_prob=SELFPLAY_POOL_PROB, checkpoint_dir=CHECKPOINT_DIR,
                 max_selfplay_plies=MAX_SELFPLAY_PLIES,
                 value_lambda=VALUE_LAMBDA, dirichlet_alpha=DIRICHLET_ALPHA,
                 dirichlet_frac=DIRICHLET_FRAC, k_forced=K_FORCED,
                 random_pool_frac=RANDOM_POOL_FRAC)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = AZParallelSelfPlay(game, network, DEVICE,
                                   n_parallel=N_PARALLEL_GAMES,
                                   wave_per_game=WAVE_PER_GAME,
                                   fast_sims=FAST_MCTS_SIMS,
                                   full_sims=FULL_MCTS_SIMS,
                                   fast_prob=FAST_GAME_PROB,
                                   temp_threshold=TEMP_THRESHOLD,
                                   value_lambda=VALUE_LAMBDA,
                                   dirichlet_alpha=DIRICHLET_ALPHA,
                                   dirichlet_frac=DIRICHLET_FRAC,
                                   k_forced=K_FORCED,
                                   pool_prob=SELFPLAY_POOL_PROB,
                                   checkpoint_dir=CHECKPOINT_DIR,
                                   channels=CHANNELS, num_blocks=NUM_BLOCKS,
                                   max_selfplay_plies=MAX_SELFPLAY_PLIES,
                                   random_pool_frac=RANDOM_POOL_FRAC)
episode_stream = self_play.episodes()
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = AlphaZeroChessNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

replay_buffer = []
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# ONE shared Elo table keyed by player-key (checkpoint × sim-budget); 'random'
# is a single player. Keys: 'random', and f'{label}@{sims}' per benchmark×sims.
def _pkey(label, sims):
    return 'random' if label == 'random' else f'{label}@{sims}'

def _eval_players():
    """Build the current eval-player pool from `benchmarks` × EVAL_SIMS +
    random. Each carries its checkpoint net and sim budget."""
    players = [{'key': 'random', 'label': 'random', 'sims': 0, 'net': None}]
    for b in benchmarks:
        if b['label'] == 'random':
            continue
        for s in EVAL_SIMS:
            players.append({'key': _pkey(b['label'], s), 'label': b['label'],
                            'sims': s, 'net': b['net']})
    return players

elos = {'random': START_ELO}
wdl  = defaultdict(lambda: [0, 0, 0])
hist = {'ep': [], 'loss': [], 'pol_loss': [], 'val_loss': [],
        'draw_pct': [], 'cut_pct': [], 'dec_pct': [], 'mean_plies': [],
        'buf_total': [], 'buf_solved': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}
_prev_stats = {}

# ── Checkpointing ─────────────────────────────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep, 'model': _model_sd, 'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': dict(elos),
                'wdl': [[ka, kb, c[0], c[1], c[2]] for (ka, kb), c in wdl.items()],
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    try:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    except Exception:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=False)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    hist = _ck['hist']
    _npad = len(hist['ep'])
    for _key in ('pol_loss', 'val_loss', 'draw_pct', 'cut_pct', 'dec_pct',
                 'mean_plies', 'buf_total', 'buf_solved'):
        hist.setdefault(_key, [float('nan')] * _npad)
    benchmarks = [{'label': 'random', 'net': None}]
    for _lbl in dict.fromkeys(_ck['bench_labels']):     # dedup on resume
        if _lbl == 'random':
            continue
        _bn = AlphaZeroChessNet(CHANNELS, NUM_BLOCKS)
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    # Eval Elo pool: load the single-table format; DISCARD an older per-track
    # checkpoint's eval ratings (format changed) — training weights are intact,
    # ratings just refill from START_ELO over the next deep evals.
    _elos_ck = _ck.get('elos', {})
    if _elos_ck and not any(isinstance(v, dict) for v in _elos_ck.values()):
        elos = dict(_elos_ck)
        wdl  = defaultdict(lambda: [0, 0, 0])
        for _row in _ck.get('wdl', []):
            if isinstance(_row, (list, tuple)) and len(_row) == 5:
                wdl[(_row[0], _row[1])] = [_row[2], _row[3], _row[4]]
    else:                                    # old per-track ratings → reset
        elos = {'random': START_ELO}
        wdl  = defaultdict(lambda: [0, 0, 0])
        hist['deep_ep'] = []; hist['elo_snap'] = []
    for _p in _eval_players():               # ensure every current key is rated
        elos.setdefault(_p['key'], START_ELO)
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_eval(new_keys):
    """Rate the current pool after adding `new_keys`, then snapshot the table."""
    players = _eval_players()
    for _p in players:
        elos.setdefault(_p['key'], START_ELO)
    run_eval_pool(players, elos, wdl, game, EVAL_DEVICE, new_keys,
                  k=K_FACTOR, games_per_pair=EVAL_GAMES_PER_PAIR,
                  random_mult=EVAL_RANDOM_MULT, opening_plies=EVAL_OPENING_PLIES)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append(dict(elos))


def _print_ladder(prefix, top=16):
    players = _eval_players()
    order = sorted(players, key=lambda p: -elos[p['key']])
    ladder = '  '.join(f'{p["key"]}={elos[p["key"]]:.0f}' for p in order[:top])
    more = f'  …(+{len(order) - top} more)' if len(order) > top else ''
    print(f'{prefix} pool Elos (top {min(top, len(order))}): {ladder}{more}')
    # Current model's search benefit, if it is in the pool.
    cur = str(ep_now[0]) if ep_now[0] > 0 else '0'
    if _pkey(cur, EVAL_SIMS[0]) in elos:
        by_sim = '  '.join(f'@{s}={elos[_pkey(cur, s)]:.0f}' for s in EVAL_SIMS)
        print(f'{prefix} current model ({cur}) by sims: {by_sim}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  AlphaZeroChessNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | PUCT + MCTS-Solver + KataGo '
      f'(forced-playout k={K_FORCED}, target pruning, value λ={VALUE_LAMBDA}) | '
      f'pool {SELFPLAY_POOL_PROB:.0%}(rand {RANDOM_POOL_FRAC:.0%}) | no aug | '
      f'{TRAIN_STEPS_PER_EP} steps/ep | ply cap {MAX_SELFPLAY_PLIES}')
print(f'Quick eval every {EVAL_EVERY} eps; deep eval-pool every {DEEP_EVAL_EVERY} '
      f'eps | sim budgets: {EVAL_SIMS} (one shared Elo table)')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} '
          f'games (wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

aux_total = 0
ep_now = [start_ep - 1]
if start_ep == 1:
    _save_benchmark_net('0', _init_snap)
    for _p in _eval_players():
        elos.setdefault(_p['key'], START_ELO)
    _run_eval([_pkey('0', s) for s in EVAL_SIMS])
    _print_ladder('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

_A = _NUM_ACTIONS
for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game (samples + solver-labeled aux).
    network.eval()
    raw_data = next(episode_stream)
    _n_aux_ep = int(getattr(self_play, 'last_aux', 0))
    aux_total += _n_aux_ep

    # 2. Buffer insertion. A sample is (obs fp16, action_idx, policy, value,
    #    solved). The solver-labeled aux samples are the LAST _n_aux_ep of the
    #    episode — tag them so the buffer solved-fraction is a live diagnostic.
    _n_ep = len(raw_data)
    for _k, (obs, ai, pol, val) in enumerate(raw_data):
        replay_buffer.extend(
            augment_sample(obs, ai, pol, val, solved=(_k >= _n_ep - _n_aux_ep)))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train: AlphaZero policy CE + value MSE on dense per-batch targets.
    network.train()
    losses, pol_losses, val_losses = [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            mask_np = np.zeros((BATCH_SIZE, _A), np.float32)
            pol_np  = np.zeros((BATCH_SIZE, _A), np.float32)
            val_np  = np.empty(BATCH_SIZE, np.float32)
            for _b, (_o, _ai, _pol, _val, _sv) in enumerate(batch):
                mask_np[_b, _ai] = 1.0
                pol_np[_b, _ai]  = _pol
                val_np[_b]       = _val
            with MODEL_LOCK:
                x_b   = batch_to_tensor([s[0] for s in batch], DEVICE)
                mask  = torch.from_numpy(mask_np).to(DEVICE)
                pol_t = torch.from_numpy(pol_np).to(DEVICE)
                val_t = torch.from_numpy(val_np).to(DEVICE)
                logits, value = network(x_b)
                optimizer.zero_grad()
                loss, pl, vl = policy_value_loss(logits, value, mask, pol_t, val_t)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()
            losses.append(float(loss.item()))
            pol_losses.append(float(pl.item()))
            val_losses.append(float(vl.item()))
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica).
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mloss = float(np.mean(losses))     if losses     else float('nan')
    mpol  = float(np.mean(pol_losses)) if pol_losses else float('nan')
    mval  = float(np.mean(val_losses)) if val_losses else float('nan')
    # Self-play outcome mix over this eval window (diff the cumulative counters).
    _st  = dict(getattr(self_play, 'stats', {}))
    _dg  = _st.get('games', 0)   - _prev_stats.get('games', 0)
    _dd  = _st.get('draw', 0)    - _prev_stats.get('draw', 0)
    _dc  = _st.get('cutoff', 0)  - _prev_stats.get('cutoff', 0)
    _dp  = _st.get('plies', 0)   - _prev_stats.get('plies', 0)
    _prev_stats = _st
    draw_pct = 100.0 * _dd / _dg if _dg else float('nan')
    cut_pct  = 100.0 * _dc / _dg if _dg else float('nan')
    dec_pct  = 100.0 * (_dg - _dd - _dc) / _dg if _dg else float('nan')
    mean_plies = _dp / _dg if _dg else float('nan')
    buf_total  = len(replay_buffer)
    buf_solved = sum(1 for _s in replay_buffer if _s[4])
    _slv_pct = 100.0 * buf_solved / buf_total if buf_total else 0.0
    hist['ep'].append(ep); hist['loss'].append(mloss)
    hist['pol_loss'].append(mpol); hist['val_loss'].append(mval)
    hist['draw_pct'].append(draw_pct); hist['cut_pct'].append(cut_pct)
    hist['dec_pct'].append(dec_pct);   hist['mean_plies'].append(mean_plies)
    hist['buf_total'].append(buf_total); hist['buf_solved'].append(buf_solved)
    _diag = (f'pol={mpol:.3f} val={mval:.3f} dr={draw_pct:.0f}% cut={cut_pct:.0f}% '
             f'ply={mean_plies:.0f} buf={buf_total/1000:.0f}k/{_slv_pct:.0f}%slv '
             f'aux={aux_total}')
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        new_label = str(ep)
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        _existing = next((b for b in benchmarks if b['label'] == new_label), None)
        if _existing is not None:
            _existing['net'] = snap
        else:
            benchmarks.append({'label': new_label, 'net': snap})
        # Warm-start each new player's Elo from the previous generation's same-
        # sim rating (falls back to START_ELO), then rate the whole pool.
        _prev = benchmarks[-2]['label'] if len(benchmarks) >= 2 else 'random'
        _new_keys = []
        for s in EVAL_SIMS:
            nk = _pkey(new_label, s)
            elos[nk] = elos.get(nk, elos.get(_pkey(_prev, s), START_ELO))
            _new_keys.append(nk)
        _run_eval(_new_keys)
        print(f'Ep {ep:5d} | loss={mloss:.3f} {_diag} '
              f'| DEEP eval-pool ({len(_eval_players())} players) | lr={lr_now:.2e}')
        _print_ladder('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        print(f'Ep {ep:5d} | loss={mloss:.3f} {_diag} '
              f'| vs {ref["label"]:>5} W{w:2d}D{d:2d}L{l:2d} '
              f'(@0 Elo={elos.get(_pkey(ref["label"], 0), START_ELO):.0f}) '
              f'| lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
fig.suptitle('Chess AlphaZero Training (control for ThompsonZero)', fontsize=14)

# Losses: total, policy cross-entropy, value MSE.
ax = axes[0, 0]
ax.plot(hist['ep'], hist['loss'], color='tab:blue', label='total')
ax.plot(hist['ep'], hist['pol_loss'], color='tab:orange', label='policy CE')
ax.plot(hist['ep'], hist['val_loss'], color='tab:green', label='value MSE')
ax.set_title('Training loss'); ax.set_xlabel('Episode'); ax.legend(fontsize=8)

# Current-model Elo by SIM BUDGET (one line per budget, from the shared pool).
def _cur_label(ep):
    return '0' if ep == 0 else str(ep)
for s in EVAL_SIMS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        key = _pkey(_cur_label(ep), s)
        if key in snap:
            xs.append(ep); ys.append(snap[key])
    axes[0, 1].plot(xs, ys, marker='.', label=f'@{s}')
axes[0, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 1].set_title('Current-model Elo by sim budget')
axes[0, 1].set_xlabel('Episode'); axes[0, 1].legend(fontsize=8)

# Self-play outcome mix — decisive% should rise, draw%/cutoff% fall.
ax = axes[0, 2]
if hist.get('draw_pct'):
    ax.plot(hist['ep'], hist['dec_pct'], color='tab:green', label='decisive %')
    ax.plot(hist['ep'], hist['draw_pct'], color='gray', label='draw %')
    ax.plot(hist['ep'], hist['cut_pct'], color='tab:red', linestyle=':', label='cutoff %')
    ax.set_ylim(0, 100); ax.set_ylabel('% of self-play games')
    axp = ax.twinx()
    axp.plot(hist['ep'], hist['mean_plies'], color='tab:blue', alpha=0.4)
    axp.set_ylabel('mean plies/game')
    ax.legend(fontsize=7, loc='center right')
ax.set_title('Self-play outcome mix'); ax.set_xlabel('Episode')

# Buffer composition: total positions + solver-labeled ground truth.
ax = axes[0, 3]
if hist.get('buf_total'):
    bt = np.array(hist['buf_total'], dtype=float)
    bs = np.array(hist['buf_solved'], dtype=float)
    ax.plot(hist['ep'], bt / 1000.0, color='tab:purple', label='buffer (k)')
    ax.plot(hist['ep'], bs / 1000.0, color='tab:orange', label='solver-labeled (k)')
    ax.set_ylabel('positions (thousands)'); ax.legend(fontsize=8)
ax.set_title('Buffer composition'); ax.set_xlabel('Episode')

# Quick-eval progress pulse: search-free W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, policy-head)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per checkpoint, grouped bars by sim budget (from the shared pool).
labels = [b['label'] for b in benchmarks if b['label'] != 'random']
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_SIMS), 1)
for k, s in enumerate(EVAL_SIMS):
    vals = [elos.get(_pkey(l, s), np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_SIMS) - 1) / 2) * width, vals, width, label=f'@{s}')
axes[1, 1].axhline(elos.get('random', START_ELO), color='gray', linestyle='--',
                   linewidth=0.8, label='random')
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per checkpoint (by sims)'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Search benefit: current-model Elo(@sims) − Elo(@0) over training. For a
# healthy AlphaZero this gap is positive and grows — search adds strength over
# the raw policy head. (This is the direct analogue of the ThompsonZero plot.)
for s in EVAL_SIMS[1:]:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        k0, ks = _pkey(_cur_label(ep), 0), _pkey(_cur_label(ep), s)
        if k0 in snap and ks in snap:
            xs.append(ep); ys.append(snap[ks] - snap[k0])
    axes[1, 2].plot(xs, ys, marker='.', label=f'@{s} − @0')
axes[1, 2].axhline(0.0, color='gray', linestyle='--', linewidth=0.8)
axes[1, 2].set_title('Search benefit (Elo over the policy head)')
axes[1, 2].set_xlabel('Episode'); axes[1, 2].set_ylabel('Elo gain from search')
axes[1, 2].legend(fontsize=8)

axes[1, 3].axis('off')
axes[1, 3].text(0.5, 0.5, 'AlphaZero control\n(vs ThompsonZero)\nsame unified eval pool',
                ha='center', va='center', fontsize=11, color='gray',
                transform=axes[1, 3].transAxes)

plt.tight_layout()
plt.show()

## Arena: pit any two checkpoints against each other

There's no existing "KataZero-chess" notebook to compare against the way the
Boop notebooks compare AlphaZeroZero to KataZero, so this section is a general
**checkpoint-vs-checkpoint arena** instead: load any two `AlphaZeroChessNet`
state dicts (any `bench_<ep>.pt` from `CHECKPOINT_DIR`, or `latest.pt`'s
`['model']` entry) and play them off, search-free or with search, alternating
colours, with a printable move-by-move game log (chess positions are
human-inspectable via FEN/SAN, unlike Boop's board).

The training cell's own DEEP-eval tournaments already give you an ongoing Elo
ladder across generations (that *is* the primary strength signal — see the
plots cell); this section is for **ad-hoc** comparisons: two specific
checkpoints, a specific `LOSS_FN` variant against another, or a specific sim
budget.

**Plugging in an external opponent** (Stockfish via UCI, another engine, a
human): `arena()` below is specifically for two `AlphaZeroChessNet` checkpoints.
To face something else, write your own move loop around a `pick(state) ->
action` function for the external side (any `state.legal_actions()`-compatible
choice) — `arena()`'s body is short and is a template for that; nothing here
has been tested against an external engine, so treat this as a starting point,
not a ready integration.

In [ ]:
import os


def _load_sd(path):
    # Full training checkpoints (latest.pt) also store non-tensor objects — e.g.
    # a numpy float in the LR-scheduler state (np.cos in the LR schedule) — which
    # torch's default weights_only=True unpickler rejects. Try the safe load,
    # then fall back to a full load for these trusted, locally-produced files.
    try:
        sd = torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)
    return sd['model'] if isinstance(sd, dict) and 'model' in sd else sd


def load_chess_net(path, channels=CHANNELS, num_blocks=NUM_BLOCKS):
    net = AlphaZeroChessNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


def arena(net_a, net_b, game, n_games=20, a_sims=0, b_sims=0,
          device='cpu', opening_plies=4, verbose=True):
    """net_a vs net_b. sims == 0 -> search-free (prior-mean argmax); sims > 0
    -> full AlphaZero-MCTS search (most-visited move, no root noise needed).
    Alternates colours; the first `opening_plies` moves of each game are random
    for variety. Returns (wins_a, draws, wins_b) from net_a's perspective."""
    bot_a = (make_az_bot(game, net_a, device, a_sims,
                               batch_size=max(1, min(a_sims, 16)))
             if a_sims > 0 else None)
    bot_b = (make_az_bot(game, net_b, device, b_sims,
                               batch_size=max(1, min(b_sims, 16)))
             if b_sims > 0 else None)
    w = d = l = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2                 # alternate colours
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())
            elif state.current_player() == a_player:
                action = (_mcts_move(bot_a, state) if bot_a is not None
                          else policy_action(net_a, state, device))
            else:
                action = (_mcts_move(bot_b, state) if bot_b is not None
                          else policy_action(net_b, state, device))
            state.apply_action(action)
            ply += 1
        # A non-terminal state's returns() is exactly [0.0, 0.0] in chess, so a
        # ply-cap cutoff scores as a draw here with no special-casing needed.
        r = state.returns()[a_player]
        if r > 0:   w += 1
        elif r < 0: l += 1
        else:       d += 1
        if verbose and (i + 1) % 5 == 0:
            print(f'    {i + 1:3d}/{n_games}  net_a W{w} D{d} L{l}')
    return w, d, l


def play_and_show(net_a, net_b, game, device='cpu', max_plies=200):
    """One game, net_a (White) vs net_b (Black), search-free, printing each
    move in SAN and the final result — a quick sanity check you can eyeball.
    Either net may be None for a random mover (e.g. no second checkpoint yet)."""
    state = game.new_initial_state()
    moves = []
    ply = 0
    while not state.is_terminal() and ply < max_plies:
        mover = state.current_player()
        net = net_a if mover == 1 else net_b     # player 1 = White (see cell 2)
        action = (random.choice(state.legal_actions()) if net is None
                  else policy_action(net, state, device))
        moves.append(state.action_to_string(mover, action))
        state.apply_action(action)
        ply += 1
    print(' '.join(f'{i//2+1}.{m}' if i % 2 == 0 else m
                   for i, m in enumerate(moves)))
    if state.is_terminal():
        print('Result:', state.returns(), '  FEN:', state)
    else:
        print(f'(stopped at the {max_plies}-ply display cap, not terminal)')
    return state


# ── Run it ────────────────────────────────────────────────────────────────────
# By default: the LATEST checkpoint vs the UNTRAINED episode-0 net and vs
# random, using whatever this session has already trained/loaded. Point
# CKPT_A / CKPT_B at any two bench_<ep>.pt files (or another CHECKPOINT_DIR
# entirely) to compare specific generations or LOSS_FN variants.
CKPT_A       = os.path.join(CHECKPOINT_DIR, 'latest.pt')   # newest
CKPT_B       = os.path.join(CHECKPOINT_DIR, 'bench_0.pt')  # untrained baseline
ARENA_GAMES  = 20
ARENA_SIMS   = 50     # per-move simulations for the with-search match
ARENA_DEVICE = 'cpu'  # batch-1 inference: CPU beats DirectML here

if not os.path.exists(CKPT_A):
    print(f'No checkpoint at {CKPT_A} yet — train at least one episode first.')
else:
    net_a = load_chess_net(CKPT_A)
    net_b = load_chess_net(CKPT_B) if os.path.exists(CKPT_B) else None
    print(f'net_a: {CKPT_A}')
    print(f'net_b: {CKPT_B if net_b is not None else "(missing — using random)"}')

    if net_b is not None:
        print(f'\nSearch-free ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES, 0, 0, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

        print(f'\n{ARENA_SIMS}-sim MCTS ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES,
                        ARENA_SIMS, ARENA_SIMS, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

    print('\nOne search-free game vs random, shown move-by-move:')
    play_and_show(net_a, None if net_b is None else net_b, game, ARENA_DEVICE)